# VSCS dense reconstruction on Colab (P1-T7)

Runs COLMAP dense stereo on the scan's sparse model. This stage lives on Colab because `patch_match_stereo` needs a CUDA GPU and the laptop has none (ADR 0003).

> **No logic here** (CLAUDE.md §4.3): install, mount, call `scripts/dense.py`.

**Before running:** upload to Drive the P1-T5 recon run folder (from `scripts/recon.py`) and the frames folder it was built from (from `scripts/extract_frames.py`).

**Budget (CLAUDE.md §1):** T4 runtime. A few hundred frames at `max_image_size: 1600` is expected to take on the order of an hour, but this has not been measured yet. Note the actual time and compute units in the devlog; warn before anything likely to exceed ~10 CU.

## 1. GPU check

In [ ]:
import subprocess

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

## 2. Mount Drive and point at the inputs

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path

VSCS_DRIVE = Path("/content/drive/MyDrive/vscs")
# Edit these two to the folders you uploaded:
RECON_RUN = VSCS_DRIVE / "data/processed/recon/<YYYYMMDD-HHMM_sha>"
IMAGES = VSCS_DRIVE / "data/processed/capture/<YYYYMMDD-HHMM_sha>/frames"
OUT_ROOT = VSCS_DRIVE / "data/processed"

for p in (RECON_RUN, IMAGES):
    print(p, "OK" if p.exists() else "MISSING")

## 3. Get the code

In [ ]:
import os

REPO_URL = "https://github.com/AlvinKalaivannan/VSCS-Vehicle-Scanning-Collision-System-.git"
REPO_DIR = Path("/content/vscs")
if not REPO_DIR.exists():
    !git clone --branch main {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only
os.chdir(REPO_DIR)
!pip install --quiet pyyaml "pydantic>=2" numpy opencv-python

## 4. Install a CUDA-enabled COLMAP

**Unverified - the first run decides which route works (R-12).** Ubuntu's own `colmap` package is, as far as is known, built *without* CUDA, which would fail at exactly the dense step. Two candidate routes are below; try route A first. The next cell checks the build and stops if it has no CUDA. **Record the route that worked in ADR 0003.**

In [ ]:
# Route A: conda-forge via condacolab (restarts the runtime once; re-run from the top).
# !pip install -q condacolab
# import condacolab; condacolab.install()
# !mamba install -y -c conda-forge colmap

# Route B: build from source (slow - allow 30-60 min - but reliable).
# See https://colmap.github.io/install.html#build-from-source ; build with CUDA enabled.

In [ ]:
import sys

sys.path.insert(0, str(REPO_DIR / "src"))
from vscs.recon.dense import colmap_cuda_status

status = colmap_cuda_status("colmap")
print("COLMAP CUDA status:", status)
assert status != "no_cuda", "This COLMAP has no CUDA - dense stereo cannot run. Try the other route."

## 5. Run dense reconstruction

In [ ]:
!python scripts/dense.py --recon-run "{RECON_RUN}" --images "{IMAGES}" --out-root "{OUT_ROOT}"

## 6. Before closing the tab

- [ ] `fused.ply` and `dense_result.json` are on **Drive**, not only in `/content`.
- [ ] Runtime and compute units noted in today's devlog (R-11).
- [ ] The COLMAP install route that worked is recorded in ADR 0003 (R-12).